# Analysis 20 — who talks together?

Which friends end up in the same conversation most often? (log: `analysis-log.md`, Analysis 20).

Choices so far (student): a conversation ends after 12 hours without any message in the group chat;
a person takes part with at least 1 message, and again with at least 2; the graph is built both ways and compared. A pair counts only if it shares clearly more
conversations than shuffled versions that keep each person's number of conversations and each
conversation's size.

In [ ]:
import pandas as pd

from wa_analyzer.data import load_own_chat

GAP_HOURS = 12  # silence that ends a conversation (student; was 6)
MIN_MESSAGES = 2  # messages needed to take part in a conversation (student)

own = load_own_chat().sort_values("timestamp").reset_index(drop=True)

## Gap to the previous message

How long since the message before it, whoever wrote it.

In [ ]:
# Time since the previous message, whoever wrote it (NaT for the first message)
own["time_diff"] = own["timestamp"].diff()

## Conversations

A new conversation starts where the gap is longer than `GAP_HOURS`, and at the first message.

In [ ]:
# Label conversations: a new one starts after a gap longer than GAP_HOURS.
# The first message has NaT, which is not > GAP_HOURS, so it starts conversation 0.
own["conversation_id"] = (own["time_diff"] > pd.Timedelta(hours=GAP_HOURS)).cumsum()

# Number of messages per conversation
conversation_counts = own["conversation_id"].value_counts()

## Who takes part

Messages per author per conversation; keep authors with at least `MIN_MESSAGES`.

In [ ]:
# Number of messages per author per conversation
author_counts = own.groupby(["conversation_id", "author"]).size().reset_index(name="message_count")

# Keep authors with at least MIN_MESSAGES messages in a conversation: one row per participant
participants = author_counts[author_counts["message_count"] >= MIN_MESSAGES]

## First numbers

- How many conversations?
- How many people per conversation (the distribution, not only the mean)?
- How many conversations have fewer than 2 people left (no links)?
- How many people does the 2-message rule drop, compared with 1 message?

In [ ]:
conversation_ids = own["conversation_id"].unique()
n_conversations = len(conversation_ids)

# People per conversation, with and without the 2-message rule. Conversations where nobody
# reached MIN_MESSAGES are missing from `participants`, so reindex them back in as 0.
people_any = author_counts.groupby("conversation_id").size().reindex(conversation_ids, fill_value=0)
people_min = participants.groupby("conversation_id").size().reindex(conversation_ids, fill_value=0)

print(f"Conversations: {n_conversations:,}")
print(f"Conversations with fewer than 2 participants (no links): {(people_min < 2).sum():,}")
print(
    f"Participations dropped by the {MIN_MESSAGES}-message rule: "
    f"{(people_any - people_min).sum():,} of {people_any.sum():,}"
)

# The spread, not only the mean: how many conversations have 0, 1, 2, ... participants
pd.DataFrame(
    {
        "at least 1 message": people_any.value_counts(),
        f"at least {MIN_MESSAGES} messages": people_min.value_counts(),
    }
).fillna(0).astype(int).sort_index().rename_axis("participants")

## Shared conversations per pair

For each pair of authors: in how many conversations did both take part? Built twice, with at least
1 message and with at least `MIN_MESSAGES`. These are raw counts: active people share more
conversations just by being active, so this is not yet the shuffle baseline from Stage 1.

In [ ]:
from itertools import combinations


def shared_conversations(people: pd.DataFrame) -> pd.Series:
    """Number of conversations each pair of authors took part in together."""
    pairs = [
        pair
        for _, authors in people.groupby("conversation_id")["author"]
        for pair in combinations(sorted(authors), 2)
    ]
    index = pd.MultiIndex.from_tuples(pairs, names=["author_a", "author_b"])
    return pd.Series(1, index=index).groupby(level=[0, 1]).sum()


RULES = {"at least 1 message": 1, f"at least {MIN_MESSAGES} messages": MIN_MESSAGES}
shared = pd.DataFrame(
    {rule: shared_conversations(author_counts[author_counts["message_count"] >= n]) for rule, n in RULES.items()}
).fillna(0).astype(int)

# The two rules give different totals, so compare each pair's share of all pair links, and its rank
share = shared / shared.sum()
rank = shared.rank(ascending=False, method="min").astype(int)

print(f"Pairs with at least one shared conversation: {(shared > 0).sum().to_dict()} of {len(shared)}")
print(f"Spearman rank correlation between the two rules: {shared.corr(method='spearman').iloc[0, 1]:.2f}")
share.round(3).join(rank, rsuffix=" (rank)").sort_values(list(RULES)[1], ascending=False)

## The two graphs side by side

Draft. Same circular layout and author order in both panels, so only the edges differ. Edge width is a
pair's share of all pair links in that panel, so the two panels are on the same scale.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

authors = sorted(own["author"].unique())
positions = nx.circular_layout(authors)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, rule in zip(axes, RULES):
    graph = nx.Graph()
    graph.add_nodes_from(authors)
    for (a, b), value in share[rule].items():
        if value > 0:
            graph.add_edge(a, b, weight=value)
    widths = [graph[a][b]["weight"] * 150 for a, b in graph.edges]
    nx.draw_networkx_edges(graph, positions, width=widths, alpha=0.5, ax=ax)
    nx.draw_networkx_nodes(graph, positions, node_size=300, ax=ax)
    nx.draw_networkx_labels(graph, positions, font_size=9, ax=ax)
    ax.set_title(rule)
    ax.axis("off")
plt.tight_layout()

## Shared conversations compared with chance

With the 2-message rule (student). Expected shared conversations for a pair if people joined
conversations independently: conversations of A × conversations of B ÷ conversations. Ratio = observed ÷
expected: above 1 means together more often than their activity explains.

Choice to check: "conversations" here counts only conversations where at least one person took part under
the rule. This quick version does not keep conversation sizes; the shuffle from Stage 1 does.

In [ ]:
conversations_per_author = participants.groupby("author")["conversation_id"].nunique()
n_used = participants["conversation_id"].nunique()

observed = shared[list(RULES)[1]]
a = observed.index.get_level_values("author_a")
b = observed.index.get_level_values("author_b")
expected = conversations_per_author[a].to_numpy() * conversations_per_author[b].to_numpy() / n_used
ratio = (observed / expected).rename("ratio")

print(f"Conversations used: {n_used:,}")
print(conversations_per_author.sort_values(ascending=False).to_string())
ratio.sort_values(ascending=False).round(2).to_frame()

## Heatmap

Draft. Authors ordered by hierarchical clustering on the ratio (distance = 1 ÷ ratio, average linkage), so
people who are together more often than chance sit next to each other and groups show as blocks.

In [ ]:
import numpy as np
import seaborn as sns
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform

matrix = ratio.unstack()
matrix = matrix.combine_first(matrix.T).reindex(index=authors, columns=authors)

distance = (1 / matrix).fillna(0).to_numpy()
order = [authors[i] for i in leaves_list(linkage(squareform(distance, checks=False), method="average"))]
ordered = matrix.loc[order, order]

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(
    ordered,
    mask=np.eye(len(order), dtype=bool),
    cmap="RdBu_r",
    center=1,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "shared conversations ÷ expected by chance"},
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Who ends up in the same conversations, compared with chance")
plt.tight_layout()